# Audit trace: one order, from the dashboard back to the WAL

Pick any order on the Metabase dashboard and follow it through every layer:

1. **dbt mart**: `marts.fct_orders`, joined to the SCD2 customer version valid when it was ordered
2. **warehouse silver**: `silver.orders` in Postgres, with the lineage columns the loader carried over
3. **silver Delta**: the row, plus the Delta commit (Change Data Feed) that last changed it
4. **bronze**: every CDC event for the order: Kafka topic / partition / offset, Postgres WAL LSN, writer schema id
5. **source**: the live Postgres row and the current WAL position

Run it inside the Spark image (it needs Delta, Avro and the schema archive):

```bash
make audit-trace ID=1
```

The output below is from a real run on 2026-11-02, after chaos test 05 rebuilt silver from bronze.

In [ ]:
from tidewater_spark.audit import trace_order
from tidewater_spark.config import Settings
from tidewater_spark.registry import SchemaRegistry
from tidewater_spark.schema_archive import SchemaArchive
from tidewater_spark.session import build_spark

settings = Settings.from_env()
spark = build_spark("audit-notebook", settings)
schemas = SchemaArchive(spark, settings.schema_archive_path, SchemaRegistry(settings.schema_registry_url))

In [1]:
t = trace_order(spark, settings, schemas, order_id=1)
# Pretty-printed by spark/jobs/audit_trace.py; captured output:

== 1. dbt mart (marts.fct_orders + SCD2 dim_customer as of order time)
   order_id                   1
   order_status               refunded
   order_total                717.78
   ordered_at                 2026-10-31 20:03:43.595216
   customer_sk                6f8975ba60d6ac8044e38dba586a5979
   customer_city_at_order     West Jamesside
   customer_version           1

== 2. warehouse silver (silver.orders in Postgres)
   status                     refunded
   _lsn                       425443272
   _source_topic              backfill
   _source_partition          -1
   _source_offset             -1
   _wh_loaded_at              2026-11-02 07:58:22.118976+00:00

== 3. silver Delta row
   status                     refunded
   is_deleted                 False
   _op                        backfill
   _lsn                       425443272
   _source_topic              backfill
   _source_partition          -1
   _source_offset             -1
   _silver_updated_at         2026-11-02 0

## Reading the trace

- **Events 1 to 5** (`c` then four `u`) are the order's life before the 2026-11-01 Kafka log loss, written with registry schema **9**.
- **Event 6** (`r`) is Debezium's re-snapshot after recovery: the same row re-delivered under the rebuilt registry, where `orders` is schema **7**, and ID 9 now means `order_items`. Silver decodes both correctly because bronze archives each writer schema per ingest run (ADR 0010). Note that the Kafka offsets also restarted (offset 3 appears in both runs).
- **Silver holds the backfill's version** (`_source_offset = -1`), stamped with the transaction-horizon LSN. That LSN is newer than every CDC event, so the LSN guard let it win, with identical content.
- **The warehouse** carries the same `_lsn` / offset, and the mart row is joined to the customer version valid at `ordered_at`.

Every hop is keyed by data the source itself assigned (the primary key and the WAL LSN), so the trace never depends on timestamps or guesses.